In [ ]:
import json
import os
import utils.batch_utils as bu
import utils.LLM_utils as llm
from utils.file_utils import dir_check, copy_to
from batch_eval_for_vllm import BatchAnalyzer

INFO 02-16 07:08:09 __init__.py:190] Automatically detected platform cuda.


/home/huanli/programs/miniconda3/envs/unsafe/lib/python3.10/site-packages/tree_sitter/__init__.py:36: FutureWarning: Language(path, name) is deprecated. Use Language(ptr, name) instead.
  warn("{} is deprecated. Use {} instead.".format(old, new), FutureWarning)


In [ ]:
prompt_dir = "prompt/"
sample_dir = "data/samples/"
result_root_dir = f"result/"
GPT_4 = "gpt-4o"

In [3]:
def upload_batch(analyzer: BatchAnalyzer, sample_target: str):
    batch_file = analyzer.batch_file(sample_target)
    metadata = analyzer.batch_util.upload_and_create_batch(batch_file)
    batch_id = metadata.id
    if os.path.exists(analyzer.batch_id_dict_file):
        batch_id_dict = json.load(open(analyzer.batch_id_dict_file))
    else:
        batch_id_dict = dict()
    batch_id_dict[sample_target] = batch_id
    json.dump(batch_id_dict, open(analyzer.batch_id_dict_file, "w"), indent=2)
    return metadata

def download_batch_result(analyzer: BatchAnalyzer, sample_target: str):
    batch_id_dict = json.load(open(analyzer.batch_id_dict_file))
    batch_id = batch_id_dict[sample_target]
    batch_result_file = analyzer.batch_result_file(sample_target)
    analyzer.batch_util.retrieve_batch_result(batch_id, batch_result_file)

## Comparative Experiments

In [7]:
sample_targets = ["risky", "filtered_unsafe"]

### LLM with Safe4U

In [ ]:
prompt = "Safe4U"
analyzers = []
analyzers.append(BatchAnalyzer(prompt, GPT_4))
analyzers.append(BatchAnalyzer("basic_check", GPT_4))

In [6]:
for t in sample_targets:
    for analyzer in analyzers:
        analyzer.generate_batch(t)
        upload_batch(analyzer, t)

In [8]:
# Typically, openai takes about 12 hours to process the batch
for t in sample_targets:
    for analyzer in analyzers:
        # download_batch_result(analyzer, t)
        analyzer.resolve_batch_result(t)